In [1]:
from openpyxl import load_workbook
import requests
from pathlib import Path

In [3]:
class ExcelFileDownloader:

    def __init__(self, excel_file, sheet_name, column_letter, download_folder):
        self.excel_file = excel_file
        self.sheet_name = sheet_name
        self.column_letter = column_letter
        self.download_folder = Path(download_folder)

        # create folder
        self.download_folder.mkdir(exist_ok=True)

    def load_sheet(self):
        wb = load_workbook(self.excel_file)
        ws = wb[self.sheet_name]
        return ws

    def download_file(self, url, filename):

        filepath = self.download_folder / filename

        try:
            response = requests.get(url, stream=True, timeout=60)
            response.raise_for_status()

            with open(filepath, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            print(f"Downloaded: {filename}")

        except Exception as e:
            print(f"Failed: {filename}")
            print(e)

    def process_links(self):

        ws = self.load_sheet()

        for cell in ws[self.column_letter][1:]:

            filename = cell.value

            if cell.hyperlink:

                url = cell.hyperlink.target

                self.download_file(url, filename)

            else:
                print(f"No hyperlink found for: {filename}")


In [5]:
# Create Object
downloader = ExcelFileDownloader(
    excel_file="file_links.xlsx",
    sheet_name="Sheet1",
    column_letter="A",
    download_folder="downloaded_files"
)

In [7]:
# Run
downloader.process_links()

Downloaded: RameswariMishra_Resume
